In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# ============================================================
# STAGE 1: SETUP for H2A
# ============================================================
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate peft
!git clone https://github.com/rfuentesfe/EgyptianHieroglyphicText




# Step 1: Download Franken dataset
!python /kaggle/working/EgyptianHieroglyphicText/scripts/1_download_franken_dataset.py

# Step 2: Merge both datasets
!python /kaggle/working/EgyptianHieroglyphicText/scripts/3_merge_datasets.py

# Step 3: Adjust image sizes to 224x224
!python /kaggle/working/EgyptianHieroglyphicText/scripts/4_adjust_files.py

In [8]:
import os
import shutil

franken = '/kaggle/working/GlyphFranken2025'
hiero   = '/kaggle/working/EgyptianHieroglyphicText/dataset'
merged  = '/kaggle/working/Glyph2025'

os.makedirs(merged, exist_ok=True)

def merge_into(src, dest):
    for cls in os.listdir(src):
        src_cls  = os.path.join(src, cls)
        dest_cls = os.path.join(dest, cls.lower())
        os.makedirs(dest_cls, exist_ok=True)
        for f in os.listdir(src_cls):
            src_file  = os.path.join(src_cls, f)
            dest_file = os.path.join(dest_cls, f.lower())
            if not os.path.exists(dest_file):
                shutil.copy2(src_file, dest_file)

print("Merging Franken...")
merge_into(franken, merged)
print("Merging Hiero...")
merge_into(hiero, merged)

# Verify
classes = os.listdir(merged)
total = sum(len(os.listdir(os.path.join(merged, c))) for c in classes)
print(f"\nClasses: {len(classes)}")
print(f"Total images: {total}")

Merging Franken...
Merging Hiero...

Classes: 353
Total images: 13743


In [ ]:
!python /kaggle/working/EgyptianHieroglyphicText/scripts/4_adjust_files.py

In [10]:
import os
from PIL import Image

merged = '/kaggle/working/Glyph2025'
classes = os.listdir(merged)
total = sum(len(os.listdir(os.path.join(merged, c))) for c in classes)
print(f"Classes: {len(classes)}")
print(f"Total images: {total}")

# Check a sample image size
sample_class = classes[0]
sample_img = os.listdir(os.path.join(merged, sample_class))[0]
img = Image.open(os.path.join(merged, sample_class, sample_img))
print(f"Sample image size: {img.size}")

Classes: 353
Total images: 13743
Sample image size: (224, 224)


In [13]:
import os, cv2, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

DATASET_PATH = '/kaggle/working/Glyph2025'
IMG_H, IMG_W = 224, 224
BATCH_SIZE   = 32
SEED         = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Setup done.')

Device : cuda
GPU    : Tesla T4
Setup done.


In [14]:
def enhance_contrast(image: Image.Image) -> Image.Image:
    """
    CLAHE contrast enhancement.
    Works on LAB color space to preserve colors while fixing brightness.
    Better than simple histogram equalization.
    """
    img_np = np.clip(np.array(image), 0, 255).astype('uint8')
    lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl     = clahe.apply(l)
    limg   = cv2.merge((cl, a, b))
    result = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    return Image.fromarray(result)


# ── Training transforms (strong augmentation) ───────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.Lambda(enhance_contrast),

    # Geometry augmentations
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5),
    transforms.RandomResizedCrop((IMG_H, IMG_W), scale=(0.8, 1.0)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),

    # Color / texture augmentations
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),

    # Randomly erase a small patch → forces model not to rely on one spot
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

# ── Validation transforms (no augmentation) ─────────────────
val_transforms = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.Lambda(enhance_contrast),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

inference_transforms = val_transforms 

print('Transforms defined.')

Transforms defined.


In [15]:
class SubsetWithTransform(Dataset):
    """Applies a different transform to a subset of a dataset."""
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        if self.transform:
            image = self.transform(image)
        return image, label


full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=None)
class_names  = full_dataset.classes
num_classes  = len(class_names)
print(f'Classes     : {num_classes}')
print(f'Total images: {len(full_dataset)}')
print(f'Avg per class: {len(full_dataset)/num_classes:.1f}')

n_val   = int(0.2 * len(full_dataset))
n_train = len(full_dataset) - n_val

train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_dataset = SubsetWithTransform(full_dataset, train_indices.indices, train_transforms)
val_dataset   = SubsetWithTransform(full_dataset, val_indices.indices,   val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# Save class mapping for inference / NLP pipeline
with open('class_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(full_dataset.class_to_idx, f, ensure_ascii=False, indent=2)

idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}

print(f'\nclass_mapping.json saved')
print(f'Train: {n_train} | Val: {n_val}')

Classes     : 353
Total images: 13742
Avg per class: 38.9

class_mapping.json saved
Train: 10994 | Val: 2748


In [16]:
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    ResNet50 with improved classifier head:
    - Higher dropout (0.5 / 0.3)
    - BatchNorm1d after first linear for stable training
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features  # 2048

    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),            # stabilizes training
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )
    return model


model     = build_model(num_classes, freeze_backbone=True).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,}')
print(f'Total params     : {total:,}')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 199MB/s]


Trainable params : 1,231,201
Total params     : 24,739,233


In [17]:
def mixup_data(x, y, alpha=0.2):
    """Apply MixUp to a batch. Returns mixed images and both label sets."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """MixUp-aware cross entropy loss."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('MixUp helpers ready.')

MixUp helpers ready.


In [18]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


def run_epoch(model, loader, optimizer, criterion, train=True, use_mixup=False):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0.0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if train and use_mixup:
                images, y_a, y_b, lam = mixup_data(images, labels, alpha=0.2)
                outputs = model(images)
                loss    = mixup_criterion(criterion, outputs, y_a, y_b, lam)
                preds   = outputs.argmax(1)
                correct += (lam * (preds == y_a).float() +
                           (1 - lam) * (preds == y_b).float()).sum().item()
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)
                correct += (outputs.argmax(1) == labels).sum().item()

            if train:
                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping — prevents exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total


print('Training loop defined.')

Training loop defined.


In [19]:
EPOCHS_P1  = 100
PATIENCE_P1 = 10

optimizer_p1 = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-3
)
scheduler_p1 = CosineAnnealingLR(optimizer_p1, T_max=EPOCHS_P1, eta_min=1e-5)

history_p1   = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_p1  = 0.0
no_improve   = 0

print('── Phase 1: Head only (backbone frozen) ──')
for epoch in range(EPOCHS_P1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer_p1, criterion,
                                      train=True,  use_mixup=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   optimizer_p1, criterion,
                                      train=False, use_mixup=False)
    scheduler_p1.step()

    history_p1['train_loss'].append(train_loss)
    history_p1['train_acc'].append(train_acc)
    history_p1['val_loss'].append(val_loss)
    history_p1['val_acc'].append(val_acc)

    print(f'Ep {epoch+1:03d}/{EPOCHS_P1} | '
          f'Loss {train_loss:.3f} | '
          f'Train {train_acc:.3f} | '
          f'Val {val_acc:.3f}', end='')

    if val_acc > best_val_p1:
        best_val_p1 = val_acc
        no_improve  = 0
        torch.save(model.state_dict(), 'best_resnet50_p1.pth')
        print(' ✅')
    else:
        no_improve += 1
        print(f' (no improve {no_improve}/{PATIENCE_P1})')
        if no_improve >= PATIENCE_P1:
            print(f'\n⏹ Early stopping at epoch {epoch+1}')
            break

print(f'\nPhase 1 best val accuracy: {best_val_p1:.3f}')

── Phase 1: Head only (backbone frozen) ──
Ep 001/100 | Loss 4.354 | Train 0.189 | Val 0.267 ✅
Ep 002/100 | Loss 4.031 | Train 0.233 | Val 0.278 ✅
Ep 003/100 | Loss 3.937 | Train 0.246 | Val 0.312 ✅
Ep 004/100 | Loss 3.867 | Train 0.261 | Val 0.338 ✅
Ep 005/100 | Loss 3.862 | Train 0.261 | Val 0.327 (no improve 1/10)
Ep 006/100 | Loss 3.800 | Train 0.274 | Val 0.358 ✅
Ep 007/100 | Loss 3.823 | Train 0.266 | Val 0.328 (no improve 1/10)
Ep 008/100 | Loss 3.736 | Train 0.283 | Val 0.337 (no improve 2/10)
Ep 009/100 | Loss 3.722 | Train 0.285 | Val 0.363 ✅
Ep 010/100 | Loss 3.722 | Train 0.283 | Val 0.376 ✅
Ep 011/100 | Loss 3.682 | Train 0.292 | Val 0.352 (no improve 1/10)
Ep 012/100 | Loss 3.687 | Train 0.295 | Val 0.358 (no improve 2/10)
Ep 013/100 | Loss 3.697 | Train 0.291 | Val 0.366 (no improve 3/10)
Ep 014/100 | Loss 3.662 | Train 0.294 | Val 0.357 (no improve 4/10)
Ep 015/100 | Loss 3.702 | Train 0.288 | Val 0.359 (no improve 5/10)
Ep 016/100 | Loss 3.628 | Train 0.303 | Val 0.368

In [ ]:
# Load best phase 1 checkpoint before unfreezing
model.load_state_dict(torch.load('best_resnet50_p1.pth', map_location=device, weights_only=False))  # ✅
print(f'Loaded Phase 1 best checkpoint (val={best_val_p1:.3f})')

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params now: {trainable:,}')

EPOCHS_P2   = 55
PATIENCE_P2 = 10

optimizer_p2 = AdamW(
    model.parameters(),
    lr=1e-4,           # lower LR — don't destroy pretrained weights
    weight_decay=1e-4
)
scheduler_p2 = CosineAnnealingLR(optimizer_p2, T_max=EPOCHS_P2, eta_min=1e-6)

history_p2  = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_p2 = 0.0
no_improve  = 0

print('\n── Phase 2: Full fine-tune ──')
for epoch in range(EPOCHS_P2):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer_p2, criterion,
                                      train=True,  use_mixup=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   optimizer_p2, criterion,
                                      train=False, use_mixup=False)
    scheduler_p2.step()

    history_p2['train_loss'].append(train_loss)
    history_p2['train_acc'].append(train_acc)
    history_p2['val_loss'].append(val_loss)
    history_p2['val_acc'].append(val_acc)

    print(f'Ep {epoch+1:03d}/{EPOCHS_P2} | '
          f'Loss {train_loss:.3f} | '
          f'Train {train_acc:.3f} | '
          f'Val {val_acc:.3f}', end='')

    if val_acc > best_val_p2:
        best_val_p2 = val_acc
        no_improve  = 0
        torch.save(model.state_dict(), 'best_resnet50_hieroglyph.pth')
        print(f' ✅ Best saved (val={val_acc:.3f})')
    else:
        no_improve += 1
        print(f' (no improve {no_improve}/{PATIENCE_P2})')
        if no_improve >= PATIENCE_P2:
            print(f'\n⏹ Early stopping at epoch {epoch+1}')
            break

print(f'\nPhase 2 best val accuracy: {best_val_p2:.3f}')

Loaded Phase 1 best checkpoint (val=0.388)
Trainable params now: 24,739,233

── Phase 2: Full fine-tune ──
Ep 001/55 | Loss 2.984 | Train 0.496 | Val 0.699 ✅ Best saved (val=0.699)
Ep 002/55 | Loss 2.561 | Train 0.619 | Val 0.781 ✅ Best saved (val=0.781)
Ep 003/55 | Loss 2.407 | Train 0.661 | Val 0.805 ✅ Best saved (val=0.805)
Ep 004/55 | Loss 2.316 | Train 0.687 | Val 0.837 ✅ Best saved (val=0.837)
Ep 005/55 | Loss 2.119 | Train 0.742 | Val 0.847 ✅ Best saved (val=0.847)
Ep 006/55 | Loss 2.076 | Train 0.749 | Val 0.859 ✅ Best saved (val=0.859)
Ep 007/55 | Loss 2.071 | Train 0.753 | Val 0.868 ✅ Best saved (val=0.868)
Ep 008/55 | Loss 2.050 | Train 0.756 | Val 0.868 (no improve 1/10)
Ep 009/55 | Loss 1.951 | Train 0.783 | Val 0.880 ✅ Best saved (val=0.880)
Ep 010/55 | Loss 1.879 | Train 0.796 | Val 0.883 ✅ Best saved (val=0.883)
Ep 011/55 | Loss 1.864 | Train 0.804 | Val 0.890 ✅ Best saved (val=0.890)
Ep 012/55 | Loss 1.859 | Train 0.801 | Val 0.888 (no improve 1/10)
Ep 013/55 | Loss 1.

In [ ]:
def plot_history(h1, h2):
    train_acc  = h1['train_acc']  + h2['train_acc']
    val_acc    = h1['val_acc']    + h2['val_acc']
    train_loss = h1['train_loss'] + h2['train_loss']
    val_loss   = h1['val_loss']   + h2['val_loss']
    p1_end     = len(h1['train_acc'])
    epochs     = range(1, len(train_acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(epochs, train_acc, label='Train Acc', color='royalblue', linewidth=2)
    axes[0].plot(epochs, val_acc,   label='Val Acc',   color='tomato',    linewidth=2)
    axes[0].axvline(x=p1_end, color='gray', linestyle='--', linewidth=1.5, label='Phase 1 → 2')
    axes[0].set_title('Accuracy', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Loss plot
    axes[1].plot(epochs, train_loss, label='Train Loss', color='royalblue', linewidth=2)
    axes[1].plot(epochs, val_loss,   label='Val Loss',   color='tomato',    linewidth=2)
    axes[1].axvline(x=p1_end, color='gray', linestyle='--', linewidth=1.5, label='Phase 1 → 2')
    axes[1].set_title('Loss', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(
        f'Training History  |  Best Val: {max(val_acc):.3f}',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: training_history.png')


plot_history(history_p1, history_p2)

In [ ]:
def load_best_model(path, num_classes):
    m = build_model(num_classes, freeze_backbone=False)
    m.load_state_dict(torch.load(path, map_location=device, weights_only=False))  # ✅ add this
    m.eval()
    return m.to(device)


def predict_sign(image_path: str, model: nn.Module):
    """Single image → Gardiner class + confidence + top3"""
    image  = Image.open(image_path).convert('RGB')
    tensor = val_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)
        top3_probs, top3_idx = torch.topk(probs, 3, dim=1)

    top3 = [
        (idx_to_class[i.item()], p.item())
        for i, p in zip(top3_idx[0], top3_probs[0])
    ]
    return top3[0][0], top3[0][1], top3


# Load the best saved model
best_model = load_best_model('best_resnet50_hieroglyph.pth', num_classes)
print('Best model loaded and ready for inference.')